# D? ?o?n Ngh?n C? Chai Ph?n C?ng PC - Quy tr?nh Hu?n luy?n

Notebook n?y x?y d?ng quy tr?nh hu?n luy?n m? h?nh ph?n lo?i lo?i ngh?n c? chai c?a m?y t?nh:
1. N?p v? ki?m tra d? li?u c?u h?nh ph?n c?ng.
2. Ph?n t?ch d? li?u kh?m ph? (EDA).
3. Ph?n chia t?p d? li?u hu?n luy?n v? ki?m th?.
4. Hu?n luy?n m? h?nh **Random Forest Classifier**.
5. ??nh gi? m? h?nh (Confusion Matrix, Classification Report, Feature Importance).
6. L?u m? h?nh b?ng `joblib` ?? tri?n khai tr?n Flask API.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

# Set style for plots
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. N?p v? ki?m tra d? li?u

N?p t?p d? li?u CSV ch?a 5,000 m?u c?u h?nh ph?n c?ng PC v? nh?n ph?n lo?i ngh?n c? chai t??ng ?ng.

In [ ]:
dataset_path = 'bottleneck_dataset.csv'
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Kh?ng t?m th?y file {dataset_path}. Vui l?ng ch?y generate.js ?? t?o d? li?u.")

df = pd.read_csv(dataset_path)
print(f"T?i t?p d? li?u th?nh c?ng. K?ch th??c: {df.shape}")
df.head()

## 2. Ph?n t?ch D? li?u Kh?m ph? (EDA)

Ki?m tra ph?n ph?i c?a c?c nh?n ph?n lo?i:
- `0`: C?n b?ng (Balanced - Kh?ng ngh?n)
- `1`: Ngh?n CPU (CPU Bottleneck)
- `2`: Ngh?n GPU (GPU Bottleneck)
- `3`: Ngh?n RAM (RAM Bottleneck)

In [ ]:
# Target Class Mapping for visualization labels
class_labels = {
    0: "C?n b?ng (0)",
    1: "Ngh?n CPU (1)",
    2: "Ngh?n GPU (2)",
    3: "Ngh?n RAM (3)"
}

df_viz = df.copy()
df_viz['bottleneck_label'] = df_viz['bottleneck_type'].map(class_labels)

# Class distributions
plt.figure(figsize=(8, 5))
sns.countplot(data=df_viz, x='bottleneck_label', order=class_labels.values(), palette="viridis")
plt.title("Ph?n Ph?i C?c Lo?i Ngh?n C? Chai", fontsize=14, fontweight='bold')
plt.xlabel("Lo?i Ngh?n C? Chai", fontsize=12)
plt.ylabel("S? l??ng m?u", fontsize=12)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### Th?ng k? ??c tr?ng
B?ng m? t? th?ng k? c?c thu?c t?nh s? trong t?p d? li?u.

In [ ]:
df.describe().T

## 3. Chia d? li?u Train/Test

T?ch c?c thu?c t?nh ??c tr?ng `X` v? nh?n `y`, sau ?? ph?n chia d? li?u theo t? l? 80% Train / 20% Test.

In [ ]:
feature_cols = [
    'cpu_clock_speed',
    'cpu_single_bench_score',
    'cpu_multi_bench_score',
    'ram_capacity',
    'ram_bus_speed',
    'gpu_vram',
    'gpu_bench_score',
    'gpu_time_spy',
    'gpu_time_spy_extreme',
    'gpu_fire_strike_extreme',
    'resolution_width',
    'resolution_height'
]
target_col = 'bottleneck_type'

X = df[feature_cols]
y = df[target_col]

# Perform stratified split to maintain target distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"K?ch th??c t?p Train: {X_train.shape}")
print(f"K?ch th??c t?p Test: {X_test.shape}")


## 4. Hu?n luy?n M? h?nh Random Forest

**Random Forest Classifier** l? m?t thu?t to?n h?c m?y m?nh m? thu?c nh?m Ensemble Learning (H?c k?t h?p):
- **Nguy?n l? ho?t ??ng:** Thu?t to?n x?y d?ng m?t qu?n h?p g?m nhi?u c?y quy?t ??nh ng?u nhi?n trong qu? tr?nh hu?n luy?n v? ??a ra d? ?o?n d?a tr?n k?t qu? b? phi?u ?a s? t? c?c c?y.
- **?u ?i?m:** Gi?m thi?u hi?n t??ng qu? kh?p (overfitting) nh? s? ng?u nhi?n h?a ??c tr?ng v? t?p d? li?u con, ??ng th?i cho ph?p t?nh to?n m?c ?? quan tr?ng c?a c?c ??c tr?ng ??u v?o d? d?ng.

In [ ]:
print("?ang hu?n luy?n m? h?nh Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print("Ho?n th?nh hu?n luy?n m? h?nh.")


## 5. ??nh gi? M? h?nh

Ch?ng ta ??nh gi? hi?u n?ng m? h?nh qua c?c ch? s?:
- **Accuracy (?? ch?nh x?c t?ng th?):** T? l? m?u d? ?o?n ??ng tr?n t?ng s? m?u ki?m th?.
- **Precision (?? ch?nh x?c d? ?o?n):** T? l? d? ?o?n ??ng l?p X tr?n t?ng s? tr??ng h?p m? h?nh d? b?o l?p X (gi?p tr?nh b?o ??ng nh?m).
- **Recall (?? ph?/?? nh?y):** T? l? ph?t hi?n ??ng l?p X tr?n t?ng s? m?u th?c t? thu?c l?p X (gi?p tr?nh b? s?t c?c ca ngh?n th?c t?).
- **F1-Score:** ?i?m trung b?nh ?i?u h?a gi?a Precision v? Recall, ??i di?n cho hi?u su?t t?ng h?a khi hai ch? s? tr?n ch?nh l?ch.

In [ ]:
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"?? ch?nh x?c tr?n t?p Test: {accuracy * 100:.2f}%\n")

target_names = [class_labels[i] for i in sorted(y.unique())]
print("B?o c?o k?t qu? ph?n lo?i (Classification Report):")
print(classification_report(y_test, y_pred, target_names=target_names))

### Ma tr?n Nh?m l?n (Confusion Matrix)
Tr?c quan h?a s? nh?m l?n gi?a nh?n d? ?o?n v? th?c t? qua b?n ?? nhi?t (heatmap).

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=target_names, yticklabels=target_names)
plt.title("B?n ?? nhi?t Ma tr?n Nh?m l?n", fontsize=13, fontweight='bold')
plt.ylabel("Nh?n Th?c t?", fontsize=11)
plt.xlabel("Nh?n D? ?o?n", fontsize=11)
plt.tight_layout()
plt.show()

### ?? quan tr?ng c?a c?c ??c tr?ng (Feature Importance)
??nh gi? c?c thu?c t?nh ??u v?o ??ng vai tr? quan tr?ng th? n?o trong quy?t ??nh c?a m? h?nh.

In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Create plot
plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices] * 100, y=[feature_cols[i] for i in indices], palette="magma")
plt.title("M?c ?? Quan Tr?ng C?a C?c ??c Tr?ng", fontsize=13, fontweight='bold')
plt.xlabel("T? l? ??ng g?p (%)", fontsize=11)
plt.ylabel("Thu?c t?nh ??c tr?ng", fontsize=11)
plt.tight_layout()
plt.show()

## 6. Xu?t M? h?nh ph?c v? Tri?n khai Web API

L?u tr? m? h?nh ?? hu?n luy?n th?nh file `.joblib` ?? server Flask c? th? n?p tr?c ti?p.

In [ ]:
model_filename = 'bottleneck_rf_model.joblib'
joblib.dump(rf_model, model_filename)
print(f"?? l?u m? h?nh th?nh c?ng t?i: '{os.path.abspath(model_filename)}'")


## 7. D? ?o?n Th? nghi?m (Sample Inference)

Ch?y d? ?o?n th? tr?n 3 c?u h?nh ki?m th? c? th? d??i c?c ?? ph?n gi?i 1080p, 1440p (2K), v? 2160p (4K):
1. **i3 12100F + 16GB RAM + RTX 5090** (CPU y?u / GPU c?c m?nh)
2. **Ryzen 7 2700X + 16GB RAM + RTX 5090** (CPU trung b?nh / GPU c?c m?nh)
3. **Ryzen 9 5900X + 32GB RAM + GTX 1060 6GB** (CPU m?nh / GPU y?u)

In [ ]:
# Load the saved model
# T?i l?i m? h?nh ?? l?u
loaded_model = joblib.load('bottleneck_rf_model.joblib')

# Class definitions mapping
bottleneck_classes = {
    0: "C?n b?ng (Kh?ng ngh?n)",
    1: "Ngh?n CPU",
    2: "Ngh?n GPU",
    3: "Ngh?n RAM"
}

# Define the 3 test configurations requested with official PassMark & 3DMark scores
test_cases = [
    {
        "name": "Intel Core i3-12100F + 16GB RAM + NVIDIA RTX 5090",
        "cpu_clock_speed": 3.3,
        "cpu_single_bench_score": 3380,       # Official PassMark single-thread avg
        "cpu_multi_bench_score": 14092,       # Official PassMark CPU Mark avg
        "ram_capacity": 16,
        "ram_bus_speed": 3200,
        "gpu_vram": 32,
        "gpu_bench_score": 19845,             # Estimated RTX 5090 Steel Nomad score
        "gpu_time_spy": 30619,
        "gpu_time_spy_extreme": 26600,
        "gpu_fire_strike_extreme": 68039
    },
    {
        "name": "AMD Ryzen 7 2700X + 16GB RAM + NVIDIA RTX 5090",
        "cpu_clock_speed": 3.7,
        "cpu_single_bench_score": 2182,       # Official PassMark single-thread avg
        "cpu_multi_bench_score": 17600,       # Official PassMark CPU Mark avg
        "ram_capacity": 16,
        "ram_bus_speed": 3000,
        "gpu_vram": 32,
        "gpu_bench_score": 19845,             # Estimated RTX 5090 Steel Nomad score
        "gpu_time_spy": 30619,
        "gpu_time_spy_extreme": 26600,
        "gpu_fire_strike_extreme": 68039     # Official 3DMark Fire Strike Extreme score
    },
    {
        "name": "AMD Ryzen 9 5900X + 32GB RAM + NVIDIA GTX 1060 6GB",
        "cpu_clock_speed": 3.7,
        "cpu_single_bench_score": 3480,       # Official PassMark single-thread avg
        "cpu_multi_bench_score": 39100,       # Official PassMark CPU Mark avg
        "ram_capacity": 32,
        "ram_bus_speed": 3600,
        "gpu_vram": 6,
        "gpu_bench_score": 930,               # Official 3DMark Steel Nomad score
        "gpu_time_spy": 4200,                 # Official 3DMark Time Spy score
        "gpu_time_spy_extreme": 1800,         # Official 3DMark Time Spy Extreme score
        "gpu_fire_strike_extreme": 5900       # Official 3DMark Fire Strike Extreme score
    }
]

# Display predictions for each resolution
resolutions = [
    {"label": "1080p Full HD", "width": 1920, "height": 1080},
    {"label": "1440p 2K QHD", "width": 2560, "height": 1440},
    {"label": "2160p 4K UHD", "width": 3840, "height": 2160}
]

for case in test_cases:
    print(f"\n================================================================================")
    print(f"C?u h?nh h? th?ng: {case['name']}")
    print(f"================================================================================")
    for res in resolutions:
        # Construct feature vector DataFrame matching columns order
        input_data = pd.DataFrame([{
            'cpu_clock_speed': case['cpu_clock_speed'],
            'cpu_single_bench_score': case['cpu_single_bench_score'],
            'cpu_multi_bench_score': case['cpu_multi_bench_score'],
            'ram_capacity': case['ram_capacity'],
            'ram_bus_speed': case['ram_bus_speed'],
            'gpu_vram': case['gpu_vram'],
            'gpu_bench_score': case['gpu_bench_score'],
            'gpu_time_spy': case['gpu_time_spy'],
            'gpu_time_spy_extreme': case['gpu_time_spy_extreme'],
            'gpu_fire_strike_extreme': case['gpu_fire_strike_extreme'],
            'resolution_width': res['width'],
            'resolution_height': res['height']
        }], columns=feature_cols)
        
        pred_val = loaded_model.predict(input_data)[0]
        prob_val = loaded_model.predict_proba(input_data)[0][pred_val]
        
        print(f"?? ph?n gi?i: {res['label']} ({res['width']}x{res['height']})")
        print(f"  -> K?t qu? d? ?o?n: {bottleneck_classes[pred_val]} (?? tin c?y: {prob_val*100:.1f}%)")
